# 1. Introduction and Project Objectives

This project implements a professional ML pipeline to predict FIFA World Cup outcomes. We focus on match-level probability estimation aggregated to tournament-level winner probabilities using historical data, advanced feature engineering, and time-aware validation.

## 2. Data Loading and Integration

We begin by loading our primary datasets:
* `results.csv`: Comprehensive international match history.
* `fifa_ranking.csv`: Longitudinal FIFA ranking data used for strength indexing.

### 2.0.1 Mount Google Drive for File Persistence

Mounting Google Drive allows us to store and access our datasets and project notebooks persistently, mirroring a local development environment. This is crucial for a multi-notebook, GitHub-ready project structure and for handling large datasets efficiently.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

# Datasets URLs (Kaggle public datasets by Mart Jürisoo)
results_url = 'https://raw.githubusercontent.com/martj42/international_results/master/results.csv'
rankings_url = 'https://raw.githubusercontent.com/martj42/international_results/master/fifa_ranking.csv'

def load_data():
    print("Loading datasets...")
    df_results = pd.read_csv(results_url)
    df_rankings = pd.read_csv(rankings_url)

    # Basic cleaning: convert dates
    df_results['date'] = pd.to_datetime(df_results['date'])
    df_rankings['rank_date'] = pd.to_datetime(df_rankings['rank_date'])

    print(f"Results loaded: {df_results.shape}")
    print(f"Rankings loaded: {df_rankings.shape}")
    return df_results, df_rankings

df_results, df_rankings = load_data()
display(df_results.head())
display(df_rankings.head())

### 2.1 Integrating Player-Level Data
To meet the requirement for player stats, ages, and playing styles, we load a dataset containing individual player attributes. This allows us to aggregate squad strengths (e.g., average rating of the starting XI).

In [ ]:
import pandas as pd

# Using a curated version of male players stats for squad aggregation
players_url = 'https://raw.githubusercontent.com/stefan-jovic/FIFA-World-Cup-2022-Prediction/main/data/players_22.csv'

def load_player_data():
    print("Loading player attributes...")
    df_players = pd.read_csv(players_url)

    # Selecting key columns for our 'Squad Strength' features later
    cols_of_interest = [
        'short_name', 'player_positions', 'overall', 'potential', 'age',
        'height_cm', 'weight_kg', 'club_name', 'league_name', 'nationality_name',
        'preferred_foot', 'weak_foot', 'skill_moves', 'work_rate', 'body_type'
    ]
    df_players = df_players[cols_of_interest]

    print(f"Player data loaded: {df_players.shape}")
    return df_players

df_players = load_player_data()
display(df_players.head())
print("Next Step: Begin Exploratory Data Analysis (EDA) to check for missing values and temporal alignment.")

### 2.2 Historical Match Data Ingestion with Temporal Guardrail

This section focuses on ingesting the `results.csv` file, applying a strict temporal filter to prevent lookahead bias (only data up to May 31, 2026), performing a structural audit, and extracting a master list of unique teams. We will also inspect the squad list file to understand its naming conventions for future entity resolution.

In [ ]:
import os
import pandas as pd
import requests
from datetime import datetime

# --- Configuration --- #
# Define the base path for our project within Google Drive
drive_project_root = '/content/drive/MyDrive/fifa-wc2026-predictor'
# UPDATED: Using the user-provided data path
data_raw_path = '/content/drive/MyDrive/ML_Assignment_Data'

# Date cutoff for lookahead bias prevention (exclusive of current WC data)
TEMPORAL_CUTOFF_DATE = datetime(2026, 5, 31)

# --- Step 1: Ensure Project Directory Structure Exists & Download results.csv ---
print(f"Ensuring directory structure exists: {data_raw_path}")
os.makedirs(data_raw_path, exist_ok=True)

# URL for the historical results, from previous cell `e94f7f23`
results_url = 'https://raw.githubusercontent.com/martj42/international_results/master/results.csv'
results_filename = 'results.csv'
results_filepath = os.path.join(data_raw_path, results_filename)

if not os.path.exists(results_filepath):
    print(f"Downloading {results_filename} to {data_raw_path}...")
    response = requests.get(results_url)
    response.raise_for_status() # Raise an exception for HTTP errors
    with open(results_filepath, 'wb') as f:
        f.write(response.content)
    print(f"Successfully downloaded {results_filename}.")
else:
    print(f"{results_filename} already exists in {data_raw_path}. Skipping download.")

# --- Step 2: Dynamic File Discovery & Ingestion ---
print("\n--- Ingesting Historical Match Data ---")
# Programmatically list files to find the results.csv
found_files = os.listdir(data_raw_path)
match_data_file = None
for f in found_files:
    if 'results' in f.lower() and f.endswith('.csv'):
        match_data_file = os.path.join(data_raw_path, f)
        break

if match_data_file is None:
    raise FileNotFoundError(f"Could not find 'results.csv' in {data_raw_path}")

print(f"Found match data file: {match_data_file}")
initial_df = pd.read_csv(match_data_file)

# Cleanly parse the match date column
initial_df['date'] = pd.to_datetime(initial_df['date'])

# --- Step 3: Lookahead Bias Defense (Strict Temporal Freeze) ---
print(f"\nApplying temporal guardrail: keeping matches played on or before {TEMPORAL_CUTOFF_DATE.strftime('%Y-%m-%d')}")
original_shape = initial_df.shape
df_matches_filtered = initial_df[initial_df['date'] <= TEMPORAL_CUTOFF_DATE].copy()

# --- Step 4: Baseline Structural Audit ---
print("\n--- Baseline Structural Audit ---")
print(f"Shape before temporal truncation: {original_shape}")
print(f"Shape after temporal truncation: {df_matches_filtered.shape}")

print(f"Verified minimum date in filtered data: {df_matches_filtered['date'].min().strftime('%Y-%m-%d')}")
print(f"Verified maximum date in filtered data: {df_matches_filtered['date'].max().strftime('%Y-%m-%d')}")

print("\nSummary of missing/null values per column:")
print(df_matches_filtered.isnull().sum())

print("\nFirst 5 rows of the filtered DataFrame:")
display(df_matches_filtered.head())

# --- Step 5: Entity Isolation (Unique Teams) ---
print("\n--- Extracting Unique Teams ---")
all_teams = pd.concat([
    df_matches_filtered['home_team'],
    df_matches_filtered['away_team']
]).unique()
unique_teams_master_list = sorted(list(all_teams))

print(f"Total unique teams found in historical data: {len(unique_teams_master_list)}")
# print(f"First 10 unique teams: {unique_teams_master_list[:10]}") # Uncomment to inspect a few

# --- Step 6: Squad File Inspection ---
print("\n--- Inspecting Squad File(s) ---")
# Try to locate a CSV file that looks like a squad list
squad_files = [f for f in found_files if 'squad' in f.lower() and f.endswith('.csv') or 'teams' in f.lower() and f.endswith('.csv')]

if squad_files:
    print(f"Found potential squad file(s): {squad_files}")
    # Assuming the first found file is the one we want to inspect
    squad_filepath = os.path.join(data_raw_path, squad_files[0])
    try:
        df_squad = pd.read_csv(squad_filepath)
        print(f"Displaying first 5 rows of {squad_files[0]}:")
        display(df_squad.head())
    except Exception as e:
        print(f"Error reading squad file {squad_files[0]}: {e}")
else:
    print(f"No obvious squad list CSV file found in {data_raw_path}. Please ensure 'FIFA World Cup 2026 Squad Lists (All 48 Teams).zip' contents are extracted.")

# Save the filtered dataframe to avoid re-processing in subsequent steps
processed_matches_filepath = os.path.join(data_raw_path, 'df_matches_filtered.csv')
df_matches_filtered.to_csv(processed_matches_filepath, index=False)
print(f"\nFiltered match data saved to: {processed_matches_filepath}")

Ensuring directory structure exists: /content/drive/MyDrive/ML_Assignment_Data
Successfully downloaded results.csv.

--- Ingesting Historical Match Data ---
Found match data file: /content/drive/MyDrive/ML_Assignment_Data/results.csv

Applying temporal guardrail: keeping matches played on or before 2026-05-31

--- Baseline Structural Audit ---
Shape before temporal truncation: (49477, 9)
Shape after temporal truncation: (49285, 9)
Verified minimum date in filtered data: 1872-11-30
Verified maximum date in filtered data: 2026-05-31

Summary of missing/null values per column:
date          0
home_team     0
away_team     0
home_score    0
away_score    0
tournament    0
city          0
country       0
neutral       0
dtype: int64

First 5 rows of the filtered DataFrame:


,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
0,1872-11-30,Scotland,England,0.0,0.0,Friendly,Glasgow,Scotland,False
1,1873-03-08,England,Scotland,4.0,2.0,Friendly,London,England,False
2,1874-03-07,Scotland,England,2.0,1.0,Friendly,Glasgow,Scotland,False
3,1875-03-06,England,Scotland,2.0,2.0,Friendly,London,England,False
4,1876-03-04,Scotland,England,3.0,0.0,Friendly,Glasgow,Scotland,False



--- Extracting Unique Teams ---
Total unique teams found in historical data: 336

--- Inspecting Squad File(s) ---
No obvious squad list CSV file found in /content/drive/MyDrive/ML_Assignment_Data. Please ensure 'FIFA World Cup 2026 Squad Lists (All 48 Teams).zip' contents are extracted.

Filtered match data saved to: /content/drive/MyDrive/ML_Assignment_Data/df_matches_filtered.csv


### 2.3 Extracting and Inspecting FIFA World Cup 2026 Squad Lists

Since the squad list files were not found directly, we will now explicitly extract the provided zip archive and then re-inspect the contents to ensure proper loading and to understand their structure for future entity resolution.

In [ ]:
import zipfile
import os
import pandas as pd

# Define the raw data path (already updated in previous cell)
data_raw_path = '/content/drive/MyDrive/ML_Assignment_Data'

# Define the path to the squad list zip file
squad_zip_filename = 'FIFA World Cup 2026 Squad Lists (All 48 Teams).zip'
squad_zip_filepath = os.path.join(data_raw_path, squad_zip_filename)

print(f"Attempting to extract: {squad_zip_filepath}")

if os.path.exists(squad_zip_filepath):
    try:
        with zipfile.ZipFile(squad_zip_filepath, 'r') as zip_ref:
            zip_ref.extractall(data_raw_path)
        print(f"Successfully extracted '{squad_zip_filename}' to '{data_raw_path}'.")

        # Now, re-inspect for squad files after extraction
        print("\n--- Re-inspecting Squad File(s) After Extraction ---")
        # Re-list files in the raw data directory
        found_files_after_extraction = os.listdir(data_raw_path)

        squad_files_extracted = [f for f in found_files_after_extraction if ('squad' in f.lower() or 'teams' in f.lower()) and f.endswith('.csv')]

        if squad_files_extracted:
            print(f"Found potential squad file(s) after extraction: {squad_files_extracted}")
            # Assuming the first found file is the one we want to inspect for now
            squad_filepath = os.path.join(data_raw_path, squad_files_extracted[0])
            try:
                df_squad = pd.read_csv(squad_filepath)
                print(f"Displaying first 5 rows of '{squad_files_extracted[0]}':")
                display(df_squad.head())
            except Exception as e:
                print(f"Error reading extracted squad file '{squad_files_extracted[0]}': {e}")
        else:
            print(f"No obvious squad list CSV file found in '{data_raw_path}' even after extracting the zip. Please check the contents of the zip file.")

    except Exception as e:
        print(f"Error extracting zip file '{squad_zip_filepath}': {e}")
else:
    print(f"Error: The zip file '{squad_zip_filepath}' was not found. Please ensure it is correctly placed in the specified directory.")

Attempting to extract: /content/drive/MyDrive/ML_Assignment_Data/FIFA World Cup 2026 Squad Lists (All 48 Teams).zip
Successfully extracted 'FIFA World Cup 2026 Squad Lists (All 48 Teams).zip' to '/content/drive/MyDrive/ML_Assignment_Data'.

--- Re-inspecting Squad File(s) After Extraction ---
Found potential squad file(s) after extraction: ['SquadLists.csv']
Displaying first 5 rows of 'SquadLists.csv':


,Team,Team Code,Number,Position,Player Name,First Name(s),Last Name(s),Name on Shirt,DOB,Club,Height (cm),Caps,Goals
0,Algeria,ALG,1,GK,MASTIL Melvin,Melvin Feycal,MASTIL,MASTIL,19/02/2000,FC Stade Nyonnais (SUI),194,2,0
1,Algeria,ALG,2,DF,MANDI Aissa,Aissa,MANDI,MANDI,22/10/1991,Lille OSC (FRA),184,119,8
2,Algeria,ALG,3,DF,ABADA Achref,Achref,ABADA,ABADA,15/06/1999,USM Alger (ALG),185,10,1
3,Algeria,ALG,4,DF,TOUGAI Mohamed Amine,Mohamed Amine,TOUGAI,TOUGAI,22/01/2000,Espérance De Tunisie (TUN),186,30,2
4,Algeria,ALG,5,DF,BELAID Zineddine,Zineddine,BELAÏD,BELAID,20/03/1999,JS Kabylie (ALG),186,18,1


### 2.4 Entity Resolution and Country Name Standardization

This section addresses the crucial task of reconciling team names between the `df_squad` (2026 World Cup teams) and `df_matches_filtered` (historical matches) dataframes. Due to variations in naming conventions, an automated audit, fuzzy matching, and a manual mapping process will be applied to ensure consistent entity identification. Finally, a summary of historical match data for the 48 World Cup teams will be generated.

In [ ]:
import difflib
import pandas as pd

# --- Step 1: Extract World Cup Contenders ---
# This list will hold the ORIGINAL names from df_squad before any mapping.
wc_2026_teams_original = df_squad['Team'].unique().tolist()
print(f"Extracted {len(wc_2026_teams_original)} unique World Cup 2026 teams.\n")

# Assert that we have exactly 48 teams as expected
assert len(wc_2026_teams_original) == 48, f"Expected 48 World Cup teams, but found {len(wc_2026_teams_original)}."

# --- Step 2: Automated Mismatch Audit ---
mismatched_wc_teams = []
print("--- Mismatch Audit: World Cup Teams vs. Historical Master List ---")
for team in wc_2026_teams_original:
    if team not in unique_teams_master_list:
        mismatched_wc_teams.append(team)
        print(f"  Mismatch found: '{team}' not in historical master list.")

if not mismatched_wc_teams:
    print("  No mismatches found initially. All 48 teams have exact historical matches.")
else:
    print(f"\nTotal {len(mismatched_wc_teams)} World Cup teams with no exact match in historical data.\n")

# --- Step 3: Fuzzy Matching Suggestion Engine ---
print("-- Fuzzy Matching Suggestions for Mismatched Teams ---")
for team in mismatched_wc_teams:
    # Using a lower cutoff as 'Côte d'Ivoire' had no matches previously
    closest_matches = difflib.get_close_matches(team, unique_teams_master_list, n=3, cutoff=0.5)
    print(f"  '{team}' (WC): Closest historical matches: {closest_matches}")

# --- Step 4: Clean Integration Map (Manual Mapping based on audit and fuzzy matches) ---
# This dictionary is explicitly defined based on expected common variations and
# insights gained from the fuzzy matching step. The key should match the exact problematic string.
TEAM_NAME_MAPPING = {
    'Bosnia And Herzegovina': 'Bosnia and Herzegovina',
    'Cabo Verde': 'Cape Verde',
    'Congo DR': 'DR Congo',
    # CRITICAL FIX: The key must exactly match the problematic string from df_squad,
    # and the VALUE must match the canonical name in historical data.
    "Côte d'Ivoire": "Ivory Coast", # Corrected: Mapping 'Côte d'Ivoire' to 'Ivory Coast'
    'Czechia': 'Czech Republic',
    'IR Iran': 'Iran',
    'Korea Republic': 'South Korea',
    'Türkiye': 'Turkey',
    'USA': 'United States'
}

print("\n--- Applying Team Name Mapping ---")
# Apply mapping to df_squad
df_squad['Team'] = df_squad['Team'].replace(TEAM_NAME_MAPPING)
print("  'df_squad' 'Team' column updated.")

# Apply mapping to df_matches_filtered
df_matches_filtered['home_team'] = df_matches_filtered['home_team'].replace(TEAM_NAME_MAPPING)
df_matches_filtered['away_team'] = df_matches_filtered['away_team'].replace(TEAM_NAME_MAPPING)
print("  'df_matches_filtered' 'home_team' and 'away_team' columns updated.")

# --- Crucial Fix: Regenerate unique_teams_master_list from the UPDATED df_matches_filtered ---
print("\n--- Regenerating Unique Teams Master List from Updated Historical Data ---")
all_teams_updated = pd.concat([
    df_matches_filtered['home_team'],
    df_matches_filtered['away_team']
]).unique()
unique_teams_master_list_updated = sorted(list(all_teams_updated))

print(f"Regenerated master list contains {len(unique_teams_master_list_updated)} unique teams.")

# --- Step 5: Sanity Verification ---
print("\n--- Re-running Mismatch Audit for Sanity Check ---")

# RECREATE wc_2026_teams FROM THE UPDATED df_squad for sanity check
wc_2026_teams_after_mapping = df_squad['Team'].unique().tolist()

mismatched_wc_teams_after_mapping = []
for team in wc_2026_teams_after_mapping:
    # Compare against the newly updated historical master list
    if team not in unique_teams_master_list_updated:
        mismatched_wc_teams_after_mapping.append(team)

if not mismatched_wc_teams_after_mapping:
    print("  Success! All World Cup 2026 teams now have exact matches in the historical master list.")
else:
    print(f"  Error: {len(mismatched_wc_teams_after_mapping)} mismatches still remain after mapping: {mismatched_wc_teams_after_mapping}")

# --- Step 6: Initial EDA Summary for World Cup Teams ---
print("\n--- Initial EDA Summary for World Cup 2026 Teams (Historical Data) ---")

# Filter matches to include only the 48 WC teams using the *mapped* team names
wc_historical_matches = df_matches_filtered[
    (df_matches_filtered['home_team'].isin(wc_2026_teams_after_mapping)) |
    (df_matches_filtered['away_team'].isin(wc_2026_teams_after_mapping))
].copy()

# Initialize lists to store summary data
summary_data = []

for team in sorted(wc_2026_teams_after_mapping): # Iterate over the mapped team names
    team_matches = wc_historical_matches[
        (wc_historical_matches['home_team'] == team) |
        (wc_historical_matches['away_team'] == team)
    ]

    total_matches = len(team_matches)
    earliest_match = team_matches['date'].min() if total_matches > 0 else pd.NaT
    most_recent_match = team_matches['date'].max() if total_matches > 0 else pd.NaT

    summary_data.append({
        'Team': team,
        'Total Historical Matches': total_matches,
        'Earliest Match Date': earliest_match.strftime('%Y-%m-%d') if pd.notna(earliest_match) else 'N/A',
        'Most Recent Match Date': most_recent_match.strftime('%Y-%m-%d') if pd.notna(most_recent_match) else 'N/A'
    })

df_wc_team_summary = pd.DataFrame(summary_data)

# Display the summary table
display(df_wc_team_summary.set_index('Team'))

print("\nEntity resolution and initial EDA summary complete for World Cup 2026 teams.")

Extracted 48 unique World Cup 2026 teams.

--- Mismatch Audit: World Cup Teams vs. Historical Master List ---
  Mismatch found: 'Côte d'Ivoire' not in historical master list.

Total 1 World Cup teams with no exact match in historical data.

-- Fuzzy Matching Suggestions for Mismatched Teams ---
  'Côte d'Ivoire' (WC): Closest historical matches: []

--- Applying Team Name Mapping ---
  'df_squad' 'Team' column updated.
  'df_matches_filtered' 'home_team' and 'away_team' columns updated.

--- Regenerating Unique Teams Master List from Updated Historical Data ---
Regenerated master list contains 336 unique teams.

--- Re-running Mismatch Audit for Sanity Check ---
  Success! All World Cup 2026 teams now have exact matches in the historical master list.

--- Initial EDA Summary for World Cup 2026 Teams (Historical Data) ---


,Total Historical Matches,Earliest Match Date,Most Recent Match Date
Team,,,
Algeria,615,1963-01-06,2026-03-31
Argentina,1067,1902-07-20,2026-03-31
Australia,580,1922-06-17,2026-05-30
Austria,859,1902-10-12,2026-03-31
Belgium,851,1904-05-01,2026-03-31
Bosnia and Herzegovina,282,1995-11-30,2026-05-29
Brazil,1058,1914-09-20,2026-05-31
Canada,467,1885-11-28,2026-03-31
Cape Verde,234,1959-12-20,2026-05-31



Entity resolution and initial EDA summary complete for World Cup 2026 teams.
